In [53]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR
from sklearn.ensemble import StackingRegressor

from xgboost import XGBRegressor

In [37]:
df = pd.read_csv(r'data/spotify_tracks.csv')

print('shape:', df.shape)
display(df.head())

shape: (50000, 21)


,track_id,track_name,artist_name,album_name,release_year,genre,popularity,duration_ms,explicit,danceability,...,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,time_signature
0,P3fAbnFbmOHnKYaXRvj7uf,One Dance (Acoustic Version),Alex Rodriguez,The Night Album,2024,metal,14,189042,True,0.427723,...,-4.702460,0.050635,0.239506,0.181395,0.133053,0.431384,141.048735,6,0,4
1,M2wleOV911xCZkwPRQeNHp,Forever Song (Remix),Desert Wind,Burning Soul,2019,rock,11,186805,True,0.448634,...,-7.110031,0.000000,0.044463,0.097818,0.435949,0.559135,131.833287,0,1,5
2,4JSnE2NiiUHUAKw9iEU1jj,Last Mountain,The Midnight,The Night Album,2022,k-pop,23,121814,False,0.707923,...,-7.305120,0.144091,0.118380,0.000000,0.262254,0.516873,127.132954,2,1,5
3,2UVvsjaSS8VFgM0Fmxk754,Falling Star (Live),Phantom Keys,Phantom's Greatest Hits,2024,latin,34,216049,False,0.846237,...,-9.527256,0.006668,0.272844,0.000000,0.045332,0.667911,93.041715,0,1,6
4,EeVVhDIq2AnHTmt9OBGhnu,Rising Moon (feat. someone),Desert Wind,Rising Soul,2010,latin,31,156170,False,0.943720,...,-9.017653,0.057632,0.219752,0.098044,0.132083,0.772151,93.404975,7,1,4


In [38]:
X = df.drop(columns=['track_id', 'track_name', 'artist_name', 'album_name', 'popularity'])
y = df['popularity']
print('X shape:', X.shape)

X shape: (50000, 16)


In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('X train columns:', X_train.columns.tolist())

Train shape: (40000, 16)
Test shape: (10000, 16)
X train columns: ['release_year', 'genre', 'duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'key', 'mode', 'time_signature']


In [40]:
# preprocessing pipeline
def make_preprocessor(X_data):
    num_cols = X_data.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X_data.select_dtypes(include=['bool', 'str']).columns

    num_pipline = Pipeline(steps=[('scaler', StandardScaler())])

    cat_pipeline = Pipeline(steps=[('ohe', OneHotEncoder(handle_unknown='ignore'))])

    # column transformer - allows for different pipelines for different columns
    preprocessor = ColumnTransformer(transformers=[('num', num_pipline, num_cols), ('cat', cat_pipeline, cat_cols)])

    return preprocessor

preprocessor = make_preprocessor(X_train)

In [41]:
y_std = y_train.std()
print('y std:', y_std)

y std: 17.916944320439228


In [62]:
# model pipeline
def make_pipeline(model):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    return pipeline

# grouping models and their hyperparameters for grid search
configs = {
    'ElasticNet': {'pipeline': make_pipeline(ElasticNet(random_state=42)), 
                   'params': {'model__alpha': (0.01, .1, 1, 10), 'model__l1_ratio': np.linspace(0, 1, 20)}},
    

    'LinearSVR': {'pipeline': make_pipeline(LinearSVR(random_state=42, max_iter=10000)), 
            'params': {'model__C': [0.01, .1, 1, 10, 100, 300, 1000], 'model__epsilon': [0.8* y_std, 0.9 * y_std, 1 * y_std, 1.2 * y_std]}},
    

    'KNN': {'pipeline': make_pipeline(KNeighborsRegressor(p = 1, n_jobs=1, algorithm='ball_tree')), # manhattan distance 
            'params': {'model__n_neighbors': [201, 301, 401, 501], 'model__weights': ['uniform', 'distance']}},
    

    'XGBoost': {'pipeline': make_pipeline(XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators=400, learning_rate=0.05)), 
                'params': { 'model__max_depth': [1, 2, 3], 'model__subsample': [0.8, .85, 0.9, 1],
                           'model__min_child_weight': [0.3, 0.5, 1], 'model__colsample_bytree': [0.6, .8, 1]}},
    }

In [52]:
# grid search for each model
def tune_gridsearch(model_name, pipeline, params, X_data, y_data):
    print (f'Tuning {model_name}...')

    gs = GridSearchCV(estimator=pipeline, param_grid=params, cv=5, n_jobs=-1, verbose=1, scoring='neg_root_mean_squared_error')

    gs.fit(X_data, y_data)

    print(f'Best params for {model_name}:', gs.best_params_)
    print(f'Best score for {model_name}:', -gs.best_score_)
    
    return gs

gridsearch_results = {}
for model_name, config in configs.items():
    gridsearch_results[model_name] = tune_gridsearch(model_name, config['pipeline'], config['params'], X_train, y_train)

Tuning ElasticNet...
Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params for ElasticNet: {'model__alpha': 0.1, 'model__l1_ratio': np.float64(1.0)}
Best score for ElasticNet: 17.72190964317327
Tuning LinearSVR...
Fitting 5 folds for each of 28 candidates, totalling 140 fits


c:\Users\aashu\OneDrive\Desktop\Projects\spotify popularity prediction\venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best params for LinearSVR: {'model__C': 100, 'model__epsilon': np.float64(14.333555456351384)}
Best score for LinearSVR: 17.725733174025983
Tuning KNN...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params for KNN: {'model__n_neighbors': 501, 'model__weights': 'distance'}
Best score for KNN: 17.792712759602786
Tuning XGBoost...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params for XGBoost: {'model__colsample_bytree': 0.6, 'model__max_depth': 1, 'model__min_child_weight': 0.3, 'model__subsample': 1}
Best score for XGBoost: 17.724704360961915


In [60]:
# xgb early stopping
# use the best params from grid search then tune for learning rate and n_estimators

xgb_best_params_gs = gridsearch_results['XGBoost'].best_params_
xgb_best_params = {}
for param, value in xgb_best_params_gs.items():
    new_param = param.split('model__')[1]
    xgb_best_params[new_param] = value

X_xgb_train, X_xgb_val, y_xgb_train, y_xgb_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

xgb_preprocessor = make_preprocessor(X_train) # makes preprocessor object
X_xgb_train_proc = xgb_preprocessor.fit_transform(X_xgb_train) # learns proc steps + applies to train
X_xgb_val_proc = xgb_preprocessor.transform(X_xgb_val) # applies proc steps to val (does not learn from val)

xgb_es_model = XGBRegressor(objective='reg:squarederror', random_state=42, **xgb_best_params)

lr_prams = [0.01, 0.03, 0.05, 0.1]
result = []

for lr in lr_prams:
    model = XGBRegressor(objective='reg:squarederror', random_state=42, learning_rate=lr, n_estimators=2000, 
                         early_stopping_rounds=400, n_jobs=-1, **xgb_best_params)
    model.fit(X_xgb_train_proc, y_xgb_train, eval_set=[(X_xgb_val_proc, y_xgb_val)], verbose=False)

    pred_val = model.predict(X_xgb_val_proc)
    rmse = root_mean_squared_error(y_xgb_val, pred_val)
    best_iter = model.best_iteration
    result.append((lr, rmse, best_iter))

print('learning_rate, rmse, best_iter')
for lr, rmse, best_iter in result:
    print(lr, rmse, best_iter)

best_lr, best_rmse, best_iter = min(result, key=lambda x: x[1])
best_round = best_iter + 1
print('best model:', best_lr, best_rmse, best_iter)

xgbr_final = make_pipeline(XGBRegressor(objective='reg:squarederror', random_state=42, learning_rate=best_lr, 
                                        n_estimators=best_round, n_jobs=-1, **xgb_best_params))

xgbr_final = xgbr_final.fit(X_train, y_train)


learning_rate, rmse, best_iter
0.01 17.715187072753906 1066
0.03 17.715089797973633 337
0.05 17.715009689331055 161
0.1 17.71506118774414 92
best model: 0.05 17.715009689331055 161


In [ ]:
# final models
final_models = {'ElasticNet': gridsearch_results['ElasticNet'].best_estimator_,
                'LinearSVR': gridsearch_results['LinearSVR'].best_estimator_,
                'KNN': gridsearch_results['KNN'].best_estimator_,
                'XGBoost': xgbr_final}